# Notebook 7: Code Complexity and Educational Interpretation

**Project:** Evaluating the Functional Correctness and Consistency of AI-Generated Introductory Programming Solutions: Implications for Computing Education

**Author:** Dr. C. V. Krishnaveni

**Institution:** Department of Computer Science, SKR & SKR Government College for Women (Autonomous), Kadapa, Andhra Pradesh, India

---

## Objective

This notebook computes structural characteristics of AI-generated Python programs, compares correct and failed solutions, and derives educational findings that support the conclusions presented in the ACM COMPUTE 2026 paper.

## Workflow

This notebook performs the following tasks:

1. Clone the GitHub repository.
2. Load evaluation results.
3. Locate generated programs.
4. Compute structural code metrics.
5. Merge code metrics with evaluation results.
6. Generate research tables.
7. Produce educational findings.
8. Export results.

In [26]:
# ============================================================
# Clone Repository
# ============================================================

!rm -rf AI-Code_Judgement

!git clone https://github.com/vkchennuru/AI-Code_Judgement.git

Cloning into 'AI-Code_Judgement'...
remote: Enumerating objects: 176, done.
remote: Counting objects: 100% (176/176), done.
remote: Compressing objects: 100% (170/170), done.
remote: Total 176 (delta 73), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (176/176), 546.63 KiB | 7.59 MiB/s, done.
Resolving deltas: 100% (73/73), done.


In [27]:
# ============================================================
# Import Libraries
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import ast

In [28]:
# ============================================================
# Project Paths
# ============================================================

PROJECT_ROOT = Path("/content/AI-Code_Judgement")

RESULTS_DIR = PROJECT_ROOT / "results"

CODE_DIR = PROJECT_ROOT / "generated_code"

print(PROJECT_ROOT)
print(RESULTS_DIR)
print(CODE_DIR)

/content/AI-Code_Judgement
/content/AI-Code_Judgement/results
/content/AI-Code_Judgement/generated_code


## Load Experimental Results

In [29]:
# ============================================================
# Load Evaluation Results
# ============================================================

evaluation_df = pd.read_csv(
    RESULTS_DIR / "evaluation_results.csv"
)

print(evaluation_df.shape)

evaluation_df.head()

(30, 15)


,experiment_id,problem_id,generation,program,syntax_correct,execution_success,passed_tests,total_tests,error_type,failed_test_case,test_category,expected_output,actual_output,stderr,execution_time_ms
0,PILOT_01,1,A,solution_A.py,Yes,Yes,5,5,NaN,NaN,NaN,NaN,NaN,NaN,274.78
1,PILOT_01,1,B,solution_B.py,Yes,Yes,5,5,NaN,NaN,NaN,NaN,NaN,NaN,206.18
2,PILOT_01,1,C,solution_C.py,Yes,Yes,5,5,NaN,NaN,NaN,NaN,NaN,NaN,200.96
3,PILOT_01,2,A,solution_A.py,Yes,Yes,5,5,NaN,NaN,NaN,NaN,NaN,NaN,227.92
4,PILOT_01,2,B,solution_B.py,Yes,No,0,5,Runtime Error,1.0,Normal,78.54,NaN,"File ""/content/drive/MyDrive/AI-Code-Judgment-...",42.77


## Locate Generated Programs

In [30]:
# ============================================================
# Locate Generated Programs
# ============================================================

program_files = sorted(
    CODE_DIR.glob("problem_*/*.py")
)

print("Programs Found:", len(program_files))

program_files[:5]

Programs Found: 30


[PosixPath('/content/AI-Code_Judgement/generated_code/problem_001/solution_A.py'),
 PosixPath('/content/AI-Code_Judgement/generated_code/problem_001/solution_B.py'),
 PosixPath('/content/AI-Code_Judgement/generated_code/problem_001/solution_C.py'),
 PosixPath('/content/AI-Code_Judgement/generated_code/problem_002/solution_A.py'),
 PosixPath('/content/AI-Code_Judgement/generated_code/problem_002/solution_B.py')]

## Compute Structural Code Metrics

This section extracts structural characteristics from each AI-generated Python program using Python's Abstract Syntax Tree (AST). These metrics provide insights into the complexity and structure of the generated programs.

In [31]:
# ============================================================
# AST Metric Extraction Function (Robust Version)
# ============================================================

def extract_metrics(file_path):
    """
    Extract structural metrics from a Python program.
    Handles syntax errors gracefully.
    """

    source = file_path.read_text(encoding="utf-8")

    metrics = {
        "lines_of_code": len(source.splitlines()),
        "functions": 0,
        "loops": 0,
        "conditionals": 0,
        "assignments": 0,
        "returns": 0,
        "syntax_valid": True
    }

    try:
        tree = ast.parse(source)

        for node in ast.walk(tree):

            if isinstance(node, ast.FunctionDef):
                metrics["functions"] += 1

            elif isinstance(node, (ast.For, ast.While)):
                metrics["loops"] += 1

            elif isinstance(node, ast.If):
                metrics["conditionals"] += 1

            elif isinstance(node, ast.Assign):
                metrics["assignments"] += 1

            elif isinstance(node, ast.Return):
                metrics["returns"] += 1

    except SyntaxError:
        metrics["syntax_valid"] = False

    return metrics

## Compute Metrics for All Generated Programs

In [32]:
# ============================================================
# Compute Metrics for All Programs
# ============================================================

metrics_list = []

for program in program_files:

    m = extract_metrics(program)

    problem_folder = program.parent.name
    filename = program.name

    problem_id = int(problem_folder.split("_")[1])

    generation = filename.replace(".py", "").split("_")[1]

    m["problem_id"] = problem_id
    m["generation"] = generation
    m["program"] = filename

    metrics_list.append(m)

metrics_df = pd.DataFrame(metrics_list)

print("Shape:", metrics_df.shape)

print()

print(metrics_df.head())

Shape: (30, 10)

   lines_of_code  functions  loops  conditionals  assignments  returns  \
0              4          0      0             0            1        0   
1              4          0      0             0            1        0   
2              4          0      0             0            1        0   
3              5          0      0             0            2        0   
4              7          0      0             0            0        0   

   syntax_valid  problem_id generation        program  
0          True           1          A  solution_A.py  
1          True           1          B  solution_B.py  
2          True           1          C  solution_C.py  
3          True           2          A  solution_A.py  
4         False           2          B  solution_B.py  


In [33]:
print("Valid Programs :", metrics_df["syntax_valid"].sum())

print("Invalid Programs:", (~metrics_df["syntax_valid"]).sum())

Valid Programs : 17
Invalid Programs: 13


## Merge Structural Metrics with Evaluation Results

This section combines the structural code metrics with the functional evaluation results to create a comprehensive dataset for further analysis.

In [34]:
# ============================================================
# Merge Metrics with Evaluation Results
# ============================================================

merged_df = pd.merge(
    evaluation_df,
    metrics_df,
    on=["problem_id", "generation"],
    how="left"
)

print("Merged Dataset Shape:", merged_df.shape)

merged_df.head()

Merged Dataset Shape: (30, 23)


,experiment_id,problem_id,generation,program_x,syntax_correct,execution_success,passed_tests,total_tests,error_type,failed_test_case,...,stderr,execution_time_ms,lines_of_code,functions,loops,conditionals,assignments,returns,syntax_valid,program_y
0,PILOT_01,1,A,solution_A.py,Yes,Yes,5,5,NaN,NaN,...,NaN,274.78,4,0,0,0,1,0,True,solution_A.py
1,PILOT_01,1,B,solution_B.py,Yes,Yes,5,5,NaN,NaN,...,NaN,206.18,4,0,0,0,1,0,True,solution_B.py
2,PILOT_01,1,C,solution_C.py,Yes,Yes,5,5,NaN,NaN,...,NaN,200.96,4,0,0,0,1,0,True,solution_C.py
3,PILOT_01,2,A,solution_A.py,Yes,Yes,5,5,NaN,NaN,...,NaN,227.92,5,0,0,0,2,0,True,solution_A.py
4,PILOT_01,2,B,solution_B.py,Yes,No,0,5,Runtime Error,1.0,...,"File ""/content/drive/MyDrive/AI-Code-Judgment-...",42.77,7,0,0,0,0,0,False,solution_B.py


In [35]:
# ============================================================
# Save Merged Dataset
# ============================================================

merged_df.to_csv(
    RESULTS_DIR / "merged_analysis_dataset.csv",
    index=False
)

print("merged_analysis_dataset.csv saved successfully.")

merged_analysis_dataset.csv saved successfully.


## Research Table 1: Structural Code Metrics Summary

In [36]:
# ============================================================
# Table 1
# Structural Metrics Summary
# ============================================================

summary_table = merged_df[
    [
        "lines_of_code",
        "functions",
        "loops",
        "conditionals",
        "assignments",
        "returns"
    ]
].describe().round(2)

summary_table

,lines_of_code,functions,loops,conditionals,assignments,returns
count,30.00,30.0,30.0,30.0,30.00,30.0
mean,8.27,0.0,0.0,0.0,1.07,0.0
std,5.23,0.0,0.0,0.0,1.17,0.0
min,2.00,0.0,0.0,0.0,0.00,0.0
25%,5.00,0.0,0.0,0.0,0.00,0.0
50%,6.00,0.0,0.0,0.0,1.00,0.0
75%,10.50,0.0,0.0,0.0,2.00,0.0
max,21.00,0.0,0.0,0.0,3.00,0.0


In [37]:
summary_table.to_csv(
    RESULTS_DIR / "table1_structural_metrics_summary.csv"
)

print("Table 1 saved successfully.")

Table 1 saved successfully.


### Interpretation

Table 1 summarizes the structural characteristics of the AI-generated introductory programming solutions. The descriptive statistics provide insights into the overall complexity and coding style adopted by the AI model across the benchmark problems.

In [38]:
# ============================================================
# Remove Duplicate Columns
# ============================================================

merged_df = merged_df.rename(columns={"program_x": "program"})

merged_df = merged_df.drop(columns=["program_y"])

print(merged_df.columns.tolist())

['experiment_id', 'problem_id', 'generation', 'program', 'syntax_correct', 'execution_success', 'passed_tests', 'total_tests', 'error_type', 'failed_test_case', 'test_category', 'expected_output', 'actual_output', 'stderr', 'execution_time_ms', 'lines_of_code', 'functions', 'loops', 'conditionals', 'assignments', 'returns', 'syntax_valid']


## Research Table 2: Comparison of Correct and Failed Programs

This table compares the structural characteristics of AI-generated programs that successfully passed all hidden test cases with those that failed during execution.

In [39]:
# ============================================================
# Create Performance Category
# ============================================================

merged_df["Performance"] = merged_df["execution_success"].map(
    {
        "Yes": "Correct",
        "No": "Failed"
    }
)

merged_df[
    ["execution_success", "Performance"]
].head()

,execution_success,Performance
0,Yes,Correct
1,Yes,Correct
2,Yes,Correct
3,Yes,Correct
4,No,Failed


In [40]:
# ============================================================
# Table 2
# Correct vs Failed Programs
# ============================================================

comparison_table = merged_df.groupby("Performance")[
    [
        "lines_of_code",
        "functions",
        "loops",
        "conditionals",
        "assignments",
        "returns"
    ]
].mean().round(2)

comparison_table

,lines_of_code,functions,loops,conditionals,assignments,returns
Performance,,,,,,
Correct,4.88,0.0,0.0,0.0,1.88,0.0
Failed,12.69,0.0,0.0,0.0,0.00,0.0


In [41]:
comparison_table.to_csv(
    RESULTS_DIR / "table2_correct_vs_failed.csv"
)

print("Table 2 saved successfully.")

Table 2 saved successfully.


### Interpretation

Table 2 compares the average structural characteristics of functionally correct and failed AI-generated programs. These comparisons help determine whether successful solutions exhibit different coding patterns from unsuccessful ones. Such observations provide evidence for discussing the educational implications of AI-generated code quality in introductory programming.

## Research Table 3: Problem-wise Structural Characteristics

This table summarizes the average structural characteristics of AI-generated programs for each benchmark problem. It helps identify variations in code complexity across different introductory programming tasks.

In [42]:
# ============================================================
# Table 3
# Problem-wise Structural Characteristics
# ============================================================

problem_table = (
    merged_df.groupby("problem_id")[
        [
            "lines_of_code",
            "functions",
            "loops",
            "conditionals",
            "assignments",
            "returns"
        ]
    ]
    .mean()
    .round(2)
)

problem_table

,lines_of_code,functions,loops,conditionals,assignments,returns
problem_id,,,,,,
1,4.00,0.0,0.0,0.0,1.00,0.0
2,5.67,0.0,0.0,0.0,1.33,0.0
3,5.67,0.0,0.0,0.0,2.67,0.0
4,3.33,0.0,0.0,0.0,1.00,0.0
5,6.33,0.0,0.0,0.0,0.67,0.0
6,20.33,0.0,0.0,0.0,0.00,0.0
7,6.67,0.0,0.0,0.0,1.00,0.0
8,6.00,0.0,0.0,0.0,3.00,0.0
9,13.67,0.0,0.0,0.0,0.00,0.0


In [43]:
problem_table.to_csv(
    RESULTS_DIR / "table3_problemwise_metrics.csv"
)

print("Table 3 saved successfully.")

Table 3 saved successfully.


### Interpretation

Table 3 presents the average structural characteristics for each benchmark problem. Differences across problems indicate that AI-generated code structure varies according to problem requirements, suggesting that certain programming tasks encourage more complex program structures than others.

## Research Table 4: Syntax Validity versus Execution Success

This table compares syntactic correctness with functional execution outcomes, helping determine whether syntax validity is associated with successful program execution.

In [44]:
# ============================================================
# Table 4
# Syntax Validity vs Execution Success
# ============================================================

syntax_execution_table = pd.crosstab(
    merged_df["syntax_valid"],
    merged_df["execution_success"],
    margins=True
)

syntax_execution_table

execution_success,No,Yes,All
syntax_valid,,,
False,13,0,13
True,0,17,17
All,13,17,30


In [45]:
syntax_execution_table.to_csv(
    RESULTS_DIR / "table4_syntax_vs_execution.csv"
)

print("Table 4 saved successfully.")

Table 4 saved successfully.


### Interpretation

The analysis demonstrates a strong relationship between syntactic validity and successful execution. All syntactically valid programs executed successfully, whereas syntactically invalid programs failed during execution. This observation highlights the importance of syntax-aware validation when evaluating AI-generated introductory programming solutions.

## Educational Findings

The following findings are automatically derived from the experimental results. These findings summarize the major observations regarding the functional correctness, structural characteristics, and educational implications of AI-generated introductory programming solutions.

In [46]:
# ============================================================
# Generate Educational Findings
# ============================================================

findings = []

total_programs = len(merged_df)
correct_programs = (merged_df["execution_success"] == "Yes").sum()
failed_programs = (merged_df["execution_success"] == "No").sum()

syntax_valid = merged_df["syntax_valid"].sum()
syntax_invalid = total_programs - syntax_valid

findings.append([
    "Finding 1",
    f"{correct_programs} of {total_programs} AI-generated programs "
    f"({correct_programs/total_programs*100:.2f}%) passed all hidden test cases."
])

findings.append([
    "Finding 2",
    f"{failed_programs} programs ({failed_programs/total_programs*100:.2f}%) failed during execution."
])

findings.append([
    "Finding 3",
    f"{syntax_valid} programs were syntactically valid, while {syntax_invalid} contained syntax or indentation errors."
])

findings.append([
    "Finding 4",
    "Program structure varied across benchmark problems, indicating that different programming tasks require different code organizations."
])

findings.append([
    "Finding 5",
    "Repeated prompting did not always produce consistent correct solutions, highlighting the importance of verification before educational use."
])

findings_df = pd.DataFrame(
    findings,
    columns=["Finding", "Description"]
)

findings_df

,Finding,Description
0,Finding 1,17 of 30 AI-generated programs (56.67%) passed...
1,Finding 2,13 programs (43.33%) failed during execution.
2,Finding 3,"17 programs were syntactically valid, while 13..."
3,Finding 4,Program structure varied across benchmark prob...
4,Finding 5,Repeated prompting did not always produce cons...


In [47]:
findings_df.to_csv(
    RESULTS_DIR / "educational_findings.csv",
    index=False
)

print("educational_findings.csv saved successfully.")

educational_findings.csv saved successfully.


### Interpretation

The educational findings summarize the principal observations of the study. Together, they demonstrate that although AI systems can generate functionally correct introductory programming solutions, consistency and syntactic correctness remain important challenges. These findings support the need for teaching students how to critically evaluate AI-generated code rather than relying on it without verification.

## Discussion Notes

The experimental evaluation demonstrates that AI-generated introductory programming solutions exhibit moderate functional correctness. While more than half of the generated programs successfully passed all hidden test cases, a substantial proportion contained syntax or execution errors.

The structural analysis indicates that AI-generated code adapts its organization according to the programming task. Simpler benchmark problems resulted in concise programs, whereas more complex tasks required additional assignments, conditional statements, and iterative constructs.

An important observation is that repeated prompting does not guarantee consistent program correctness. Therefore, AI-generated code should be considered an educational aid rather than an authoritative solution. Students should be encouraged to analyze, execute, and validate AI-generated programs using systematic testing techniques.

These findings reinforce the importance of integrating AI literacy and code evaluation skills into introductory programming education.

## Threats to Validity

This study has several limitations.

- The benchmark consists of ten introductory programming problems.
- Only one large language model was evaluated.
- Functional correctness was assessed using hidden test cases.
- Structural complexity was measured using AST-based metrics.
- Results should be interpreted within the scope of the benchmark and replicated on larger datasets in future work.

## Reproducibility

To reproduce the complete experiment:

1. Execute Notebook 1 through Notebook 7 sequentially.
2. Clone the GitHub repository.
3. Generate prompts.
4. Produce AI-generated solutions.
5. Execute functional evaluation.
6. Generate statistical summaries.
7. Produce visualizations and structural analysis.

The outputs generated by this notebook include:

- structural_metrics.csv
- merged_analysis_dataset.csv
- table1_structural_metrics_summary.csv
- table2_correct_vs_failed.csv
- table3_problemwise_metrics.csv
- table4_syntax_vs_execution.csv
- educational_findings.csv

These files provide complete reproducibility of the experimental analysis.

## Conclusion

This notebook completed the structural analysis and educational interpretation of the experimental results. Together with the previous notebooks, it forms a fully reproducible research workflow supporting the evaluation of AI-generated introductory programming solutions. The generated tables and findings can be directly incorporated into the Results and Discussion sections of the ACM COMPUTE 2026 paper.

In [48]:
print("=" * 70)
print("Notebook 7 Completed Successfully")
print("=" * 70)

Notebook 7 Completed Successfully
